# Data Mining Lab: Book Recommender on a Medallion Data Lake
Pipeline: **Bronze** (raw + timestamp) → **Silver** (cleaned) → **Gold** (model-ready) → **Item-based collaborative filtering**.


## Step 1: Data lake setup and Bronze layer

In [1]:
import os, re, datetime
import numpy as np, pandas as pd
import scipy.sparse as sp
for d in ["bronze","silver","gold"]:
    os.makedirs(d, exist_ok=True)

# Using the CSV files you uploaded manually (BX-Users.csv, BX-Books.csv, BX-Book-Ratings.csv)
def find(name):
    for base in ["sample_data", "/content/sample_data", "/content", "."]:
        for root,_,files in os.walk(base):
            for f in files:
                if f.lower()==name.lower(): return os.path.join(root,f)
    raise FileNotFoundError(name + " not found - upload it to the Colab Files panel")

for n in ["BX-Users.csv","BX-Books.csv","BX-Book-Ratings.csv"]:
    print(n, "->", find(n))


BX-Users.csv -> sample_data/BX-Users.csv
BX-Books.csv -> sample_data/BX-Books.csv
BX-Book-Ratings.csv -> sample_data/BX-Book-Ratings.csv


In [19]:
def read_raw(name):
    # semicolon separated, latin-1 (non-UTF-8); all columns read as text so nothing is altered
    return pd.read_csv(find(name), sep=";", encoding="latin-1", dtype=str,
                       on_bad_lines="skip", engine="python", quotechar='"')

users   = read_raw("BX-Users.csv")
books   = read_raw("BX-Books.csv")
ratings = read_raw("BX-Book-Ratings.csv")

ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
for name, df in [("users",users),("books",books),("ratings",ratings)]:
    df["ingestion_timestamp"] = ts
    df.to_parquet(f"bronze/{name}.parquet", index=False)   # no cleaning in Bronze
    print(name, df.shape, list(df.columns))

users (278177, 4) ['User-ID', 'Location', 'Age', 'ingestion_timestamp']
books (161163, 9) ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L', 'ingestion_timestamp']
ratings (1149766, 4) ['User-ID', 'ISBN', 'Book-Rating', 'ingestion_timestamp']


## Step 2: EDA on Bronze data

In [3]:
users   = pd.read_parquet("bronze/users.parquet")
books   = pd.read_parquet("bronze/books.parquet")
ratings = pd.read_parquet("bronze/ratings.parquet")
# normalise column names by position (headers differ between dataset copies)
users.columns   = ["user_id","location","age","ingestion_timestamp"]
books.columns   = ["isbn","title","author","year","publisher","img_s","img_m","img_l","ingestion_timestamp"]
ratings.columns = ["user_id","isbn","rating","ingestion_timestamp"]
for n,df in [("users",users),("books",books),("ratings",ratings)]:
    print("=====",n,df.shape); print(df.dtypes.to_string()); print("missing:\n",df.isna().sum().to_string(),"\n")

===== users (278177, 4)
user_id                object
location               object
age                    object
ingestion_timestamp    object
missing:
 user_id                     0
location                    0
age                    110508
ingestion_timestamp         0 

===== books (161163, 9)
isbn                   object
title                  object
author                 object
year                   object
publisher              object
img_s                  object
img_m                  object
img_l                  object
ingestion_timestamp    object
missing:
 isbn                   0
title                  0
author                 1
year                   0
publisher              2
img_s                  0
img_m                  0
img_l                  0
ingestion_timestamp    0 

===== ratings (1149766, 4)
user_id                object
isbn                   object
rating                 object
ingestion_timestamp    object
missing:
 user_id                0
isbn       

**Shape / dtypes / missing values:** all columns were read as text in Bronze. `users` has ~278k rows with many missing ages, `books` ~271k rows with a handful of missing title/author/publisher values, `ratings` ~1.15M rows. → In Silver I convert `age`, `year`, `rating` to numeric, impute age, and drop rows missing key fields.

In [4]:
r = pd.to_numeric(ratings["rating"], errors="coerce")
print(r.value_counts().sort_index())
print("zeros: %d (%.1f%%)" % ((r==0).sum(), 100*(r==0).mean()))

rating
0     716107
1       1770
2       2759
3       5996
4       8904
5      50973
6      36922
7      76454
8     103733
9      67540
10     78608
Name: count, dtype: int64
zeros: 716107 (62.3%)


**Rating distribution:** roughly 60% of ratings are 0, which means "no explicit rating" (implicit feedback), not a bad score. → Silver removes all 0s and keeps ratings 1–10.

In [5]:
age = pd.to_numeric(users["age"], errors="coerce")
print(age.describe()); print("age<5:", (age<5).sum(), " age>100:", (age>100).sum(), " null:", age.isna().sum())
yr = pd.to_numeric(books["year"], errors="coerce")
print(yr.describe()); print("year==0:", (yr==0).sum(), " year>2026:", (yr>2026).sum(), " null:", yr.isna().sum())

count    167669.000000
mean         34.751797
std          14.430984
min           0.000000
25%          24.000000
50%          32.000000
75%          44.000000
max         244.000000
Name: age, dtype: float64
age<5: 881  age>100: 366  null: 110508
count    161163.000000
mean       1959.544945
std         260.538865
min           0.000000
25%        1990.000000
50%        1996.000000
75%        2000.000000
max        2050.000000
Name: year, dtype: float64
year==0: 2797  year>2026: 6  null: 0


**Ages:** there are impossible ages (0, 1, 2 … and values up to 200+) plus many nulls. → Silver sets <5 or >100 to null, then fills with the median.

**Publication years:** many 0 values and some future years (e.g. 2030+, 2037). → Silver sets 0 and years > current year to null.

In [6]:
per_user = ratings.groupby("user_id").size(); per_book = ratings.groupby("isbn").size()
print(per_user.describe()); print(per_book.describe())
for k in [1,5,10,20]:
    print(f"users with <{k} ratings: {(per_user<k).mean():.1%} | books with <{k} ratings: {(per_book<k).mean():.1%}")

count    105274.000000
mean         10.921652
std          90.563768
min           1.000000
25%           1.000000
50%           1.000000
75%           4.000000
max       13600.000000
dtype: float64
count    340545.000000
mean          3.376253
std          12.436447
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        2502.000000
dtype: float64
users with <1 ratings: 0.0% | books with <1 ratings: 0.0%
users with <5 ratings: 78.3% | books with <5 ratings: 87.1%
users with <10 ratings: 87.6% | books with <10 ratings: 94.6%
users with <20 ratings: 93.0% | books with <20 ratings: 97.8%


**Ratings per user / per book:** extremely long-tailed; the median user and median book have only 1–2 ratings. → Silver applies a minimum-ratings threshold (chosen in Step 3).

In [7]:
nu, nb, nr = ratings.user_id.nunique(), ratings.isbn.nunique(), len(ratings)
print("sparsity (raw, incl. zeros): %.5f%% filled" % (100*nr/(nu*nb)))
ex = ratings[pd.to_numeric(ratings.rating, errors="coerce")>0]
print("sparsity (explicit only): %.5f%% filled" % (100*len(ex)/(ex.user_id.nunique()*ex.isbn.nunique())))

sparsity (raw, incl. zeros): 0.00321% filled
sparsity (explicit only): 0.00300% filled


**Sparsity:** well under 0.01% of user-book pairs have a rating (>99.99% empty). → Use a sparse matrix (CSR) in Gold, never a dense one.

## Step 3: Silver layer

In [8]:
CURRENT_YEAR = datetime.date.today().year
MIN_USER_RATINGS = 10   # threshold, justified below
MIN_BOOK_RATINGS = 10

# ---- users
u = users.drop(columns="ingestion_timestamp").drop_duplicates()
u["user_id"] = pd.to_numeric(u["user_id"], errors="coerce")
u = u.dropna(subset=["user_id"]).drop_duplicates("user_id"); u["user_id"]=u["user_id"].astype(int)
u["age"] = pd.to_numeric(u["age"], errors="coerce")
u.loc[(u["age"]<5)|(u["age"]>100),"age"] = np.nan
u["age"] = u["age"].fillna(u["age"].median())
clean = lambda s: s.astype("string").str.replace(r"\s+"," ",regex=True).str.strip()
loc = clean(u["location"]).str.lower().str.replace('"','',regex=False)
parts = loc.str.split(",", expand=True).apply(lambda c: c.str.strip())
u["city"]    = parts[0]
u["country"] = parts[parts.columns[-1]] if parts.shape[1]>1 else pd.NA
u["state"]   = parts[1] if parts.shape[1]>2 else pd.NA
for c in ["city","state","country"]: u[c]=u[c].replace({"":pd.NA,"n/a":pd.NA})
u = u.drop(columns="location")

# ---- books
b = books.drop(columns=["ingestion_timestamp","img_s","img_m","img_l"]).drop_duplicates()
b["isbn"] = b["isbn"].astype("string").str.strip().str.upper()
b = b.drop_duplicates("isbn")
b["year"] = pd.to_numeric(b["year"], errors="coerce")
b.loc[(b["year"]==0)|(b["year"]>CURRENT_YEAR),"year"] = np.nan
b["title"] = clean(b["title"]); b["author"]=clean(b["author"]).str.title(); b["publisher"]=clean(b["publisher"]).str.title()
b = b.dropna(subset=["title","author"])

# ---- ratings
rt = ratings.drop(columns="ingestion_timestamp").drop_duplicates()
rt["user_id"] = pd.to_numeric(rt["user_id"], errors="coerce")
rt["rating"]  = pd.to_numeric(rt["rating"], errors="coerce")
rt["isbn"] = rt["isbn"].astype("string").str.strip().str.upper()
rt = rt.dropna(subset=["user_id","rating"]); rt["user_id"]=rt["user_id"].astype(int)
rt = rt[(rt.rating>=1)&(rt.rating<=10)].drop_duplicates(["user_id","isbn"])
print("explicit ratings:", len(rt))
rt = rt[rt.user_id.isin(u.user_id) & rt.isbn.isin(b.isbn)]
print("after referential integrity:", len(rt))

explicit ratings: 433659
after referential integrity: 331832


In [9]:
# Justify the threshold: how much data survives for different cut-offs (iterative filtering)
def filt(df,k):
    while True:
        n=len(df)
        df=df[df.groupby("user_id")["isbn"].transform("size")>=k]
        df=df[df.groupby("isbn")["user_id"].transform("size")>=k]
        if len(df)==n: return df
for k in [2,5,10,20,50]:
    f=filt(rt,k); print(f"k={k:>2}: ratings={len(f):>7} users={f.user_id.nunique():>6} books={f.isbn.nunique():>6}")

k= 2: ratings= 236163 users= 23818 books= 41395
k= 5: ratings= 114953 users=  6843 books=  9060
k=10: ratings=  41251 users=  1813 books=  2022
k=20: ratings=      0 users=     0 books=     0
k=50: ratings=      0 users=     0 books=     0


**Threshold choice: at least 10 ratings per user and per book.** The EDA shows most users/books have 1–2 ratings, which cannot produce a meaningful cosine similarity. A threshold of 10 removes that noise while keeping thousands of users and books (the table above shows the trade-off), so the matrix is still large enough to recommend from. Higher thresholds (20, 50) discard too much data; lower ones (2, 5) keep very noisy items.

In [10]:
rt = filt(rt, MIN_USER_RATINGS)
u = u[u.user_id.isin(rt.user_id)]; b = b[b.isbn.isin(rt.isbn)]
u.to_parquet("silver/users.parquet",index=False)
b.to_parquet("silver/books.parquet",index=False)
rt.to_parquet("silver/ratings.parquet",index=False)
print(u.shape,b.shape,rt.shape)

(1813, 5) (2022, 5) (41251, 3)


## Step 4: Gold layer

In [11]:
u  = pd.read_parquet("silver/users.parquet"); b = pd.read_parquet("silver/books.parquet"); rt = pd.read_parquet("silver/ratings.parquet")
user_ids = np.sort(rt.user_id.unique()); isbns = np.sort(rt.isbn.unique())
uidx = {x:i for i,x in enumerate(user_ids)}; bidx = {x:i for i,x in enumerate(isbns)}
M = sp.csr_matrix((rt.rating.astype(float), (rt.user_id.map(uidx), rt.isbn.map(bidx))), shape=(len(user_ids),len(isbns)))
sp.save_npz("gold/user_book_matrix.npz", M)
pd.DataFrame({"user_id":user_ids,"row":range(len(user_ids))}).to_parquet("gold/user_index.parquet",index=False)
pd.DataFrame({"isbn":isbns,"col":range(len(isbns))}).to_parquet("gold/book_index.parquet",index=False)
b[["isbn","title","author"]].rename(columns={"isbn":"book_id"}).to_parquet("gold/book_lookup.parquet",index=False)
rt.groupby("user_id").rating.agg(avg_rating="mean",num_ratings="count").reset_index().to_parquet("gold/user_aggregates.parquet",index=False)
rt.groupby("isbn").rating.agg(avg_rating="mean",num_ratings="count").reset_index().to_parquet("gold/book_aggregates.parquet",index=False)
print(M.shape, "density: %.4f%%" % (100*M.nnz/(M.shape[0]*M.shape[1])))

(1813, 2022) density: 1.1253%


## Step 5: Item-based collaborative filtering

In [12]:
from sklearn.neighbors import NearestNeighbors
M = sp.load_npz("gold/user_book_matrix.npz")
user_index = pd.read_parquet("gold/user_index.parquet"); book_index = pd.read_parquet("gold/book_index.parquet")
lookup = pd.read_parquet("gold/book_lookup.parquet").drop_duplicates("book_id").set_index("book_id")
B = M.T.tocsr()                                   # rows = books, columns = users
isbns = book_index.isbn.values; user_ids = user_index.user_id.values
uidx = {x:i for i,x in enumerate(user_ids)}; bidx = {x:i for i,x in enumerate(isbns)}
knn = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1).fit(B)
title_of = lambda i: lookup.loc[isbns[i],"title"]
titles = pd.Series({i:title_of(i) for i in range(len(isbns))})

def similar_books(title, n=10):
    hits = titles[titles.str.lower()==title.lower()]
    if hits.empty: hits = titles[titles.str.lower().str.contains(title.lower(), regex=False)]
    if hits.empty: return f"No book matching '{title}'"
    i = hits.index[0]
    d, idx = knn.kneighbors(B[i], n_neighbors=n+1)
    return pd.DataFrame({"title":[titles[j] for j in idx[0][1:]], "similarity":1-d[0][1:]})

def recommend(user_id, n=10, k=20, liked=7):
    r = uidx[user_id]; row = M.getrow(r); cols, vals = row.indices, row.data
    read = set(cols)
    fav = [(c,v) for c,v in zip(cols,vals) if v>=liked] or sorted(zip(cols,vals), key=lambda x:-x[1])[:5]
    scores = {}
    for c,v in fav:
        d, idx = knn.kneighbors(B[c], n_neighbors=min(k+1,B.shape[0]))
        for dist,j in zip(d[0][1:], idx[0][1:]):
            if j in read: continue
            scores[j] = scores.get(j,0) + (1-dist)*v          # similarity weighted by user's rating
    top = sorted(scores.items(), key=lambda x:-x[1])[:n]
    return [titles[j] for j,_ in top]

similar_books(titles.iloc[0])

,title,similarity
0,Name of the Rose-Nla,0.284382
1,The Cereal Murders,0.208070
2,"The Subtle Knife (His Dark Materials, Book 2)",0.200004
3,The Little Prince,0.157983
4,The Outsiders,0.155840
5,Devil's Waltz (Alex Delaware Novels (Paperback)),0.144334
6,Fast Food Nation: The Dark Side of the All-Ame...,0.141046
7,The Book Borrower: A Novel,0.138819
8,A Widow for One Year,0.134687
9,Nobody's Baby But Mine,0.131299


## Step 6: Demo and sanity check

In [13]:
counts = np.diff(M.indptr)
demo_users = user_ids[np.argsort(-counts)[[0,50,200]]]   # three different users (heavy to moderate readers)
for uid in demo_users:
    row = M.getrow(uidx[uid])
    liked = sorted(zip(row.indices,row.data), key=lambda x:-x[1])[:8]
    print("="*70); print("User", uid, "| books rated:", row.nnz)
    print("Highly rated:");  [print(f"  {v:.0f}  {titles[c]}") for c,v in liked]
    print("Recommended:");   [print(f"  {i+1}. {t}") for i,t in enumerate(recommend(uid,10))]

User 11676 | books rated: 950
Highly rated:
  10  The Poisonwood Bible
  10  Daughter of Fortune : A Novel (Oprah's Book Club (Hardcover))
  10  Prodigal Summer
  10  I Know This Much Is True (Oprah's Book Club)
  10  The Poisonwood Bible
  10  To Kill a Mockingbird
  10  Fast Food Nation: The Dark Side of the All-American Meal
  10  The Professor and the Madman: A Tale of Murder, Insanity, and the Making of The Oxford English Dictionary
Recommended:
  1. Night Shield (Intimate Moments, 1027)
  2. Palomino
  3. Birthright
  4. A Perfect Stranger
  5. Night Tales
  6. Public Secrets
  7. Summer's End
  8. Darkest Hour (Cutler)
  9. Stanislaski Brothers (Silhouette Promo)
  10. Witness in Death (Eve Dallas Mysteries (Paperback))
User 55490 | books rated: 66
Highly rated:
  10  Billy and the Boingers Bootleg (Bloom County Book)
  10  Cat and Mouse
  10  The Vampire Lestat (Vampire Chronicles, Book II)
  10  Sword of Shannara
  10  Dark Rivers of the Heart
  10  Pandora : New Tales of the 